<a href="https://colab.research.google.com/github/Martinz77-max/Martinz77-max/blob/main/avatarify.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Avatarify Colab Server

This Colab notebook is for running Avatarify rendering server. It allows you to run Avatarify on your computer **without GPU** in this way:

1. When this notebook is executed, it starts listening for incoming requests from your computer;
1. You start the client on your computer and it connects to the notebook and starts sending requests;
1. This notebooks receives the requests from your computer, renders avatar images and sends them back;

To this end, all the heavy work is offloaded from your computer to this notebook so you don't need to have a beafy hardware on your PC anymore.


## Start the server
Run the cells below (Shift+Enter) sequentially and pay attention to the hints and instructions included in this notebook.

At the end you will get a command for running the client on your computer.

## Start the client

Make sure you have installed the latest version of Avatarify on your computer. Refer to the [README](https://github.com/alievk/avatarify#install) for the instructions.

When it's ready execute this notebook and get the command for running the client on your computer.


### Technical details

The client on your computer connects to the server via `ngrok` TCP tunnel or a reverse `ssh` tunnel.

`ngrok`, while easy to use, can induce a considerable network lag ranging from dozens of milliseconds to a second. This can lead to a poor experience.

A more stable connection could be established using a reverse `ssh` tunnel to a host with a public IP, like an AWS `t3.micro` (free) instance. This notebook provides a script for creating a tunnel, but launching an instance in a cloud is on your own (find the manual below).

# Install

### Avatarify
Follow the steps below to clone Avatarify and install the dependencies.

In [56]:
!cd /content
!rm -rf *

In [57]:
!git clone https://github.com/alievk/avatarify.git

Cloning into 'avatarify'...
remote: Enumerating objects: 1514, done.
remote: Total 1514 (delta 0), reused 0 (delta 0), pack-reused 1514 (from 1)
Receiving objects: 100% (1514/1514), 5.69 MiB | 36.40 MiB/s, done.
Resolving deltas: 100% (967/967), done.


In [58]:
cd avatarify

/content/avatarify/avatarify


In [59]:
!git clone https://github.com/alievk/first-order-model.git fomm
!sudo apt-get install -y build-essential libyaml-dev
!pip install face-alignment==1.0.0 msgpack_numpy pyyaml

Cloning into 'fomm'...
remote: Enumerating objects: 211, done.
remote: Total 211 (delta 0), reused 0 (delta 0), pack-reused 211 (from 1)
Receiving objects: 100% (211/211), 58.16 MiB | 34.95 MiB/s, done.
Resolving deltas: 100% (108/108), done.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
build-essential is already the newest version (12.9ubuntu3).
libyaml-dev is already the newest version (0.2.2-1build2).
0 upgraded, 0 newly installed, 0 to remove and 53 not upgraded.


In [60]:
!./scripts/download_data.sh

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  228M  100  228M    0     0  75.9M      0  0:00:03  0:00:03 --:--:-- 75.9M
Expected checksum: 8a45a24037871c045fbb8a6a8aa95ebc
Found checksum:    8a45a24037871c045fbb8a6a8aa95ebc  vox-adv-cpk.pth.tar


### ngrok
Follow the steps below to setup ngrok. You will also need to sign up on the ngrok site and get your authtoken (free).


In [61]:
# Download ngrok
!scripts/get_ngrok.sh

ngrok is not found, installing...
Done!


# Run
Start here if the runtime was restarted after installation.

In [62]:
cd /content/avatarify

/content/avatarify


In [63]:
#!git pull origin

In [64]:
from subprocess import Popen, PIPE
import shlex
import json
import time

def run_with_pipe(command):
  commands = list(map(shlex.split,command.split("|")))
  ps = Popen(commands[0], stdout=PIPE, stderr=PIPE)
  for command in commands[1:]:
    ps = Popen(command, stdin=ps.stdout, stdout=PIPE, stderr=PIPE)
  return ps.stdout.readlines()


def get_tunnel_adresses():
  max_retries = 10
  retry_delay_seconds = 2
  info = None
  for i in range(max_retries):
    try:
      # Add a timeout for curl to prevent hanging indefinitely
      raw_info = run_with_pipe("curl --max-time 5 http://localhost:4040/api/tunnels")
      if raw_info:
        info = json.loads(raw_info[0])
        # Check if tunnels list is not empty or if ngrok is actually serving
        if 'tunnels' in info and len(info['tunnels']) > 0:
          break  # Success, break out of retry loop
    except json.JSONDecodeError:
      print(f"Attempt {i+1}/{max_retries}: ngrok API not ready (JSON decode error), retrying in {retry_delay_seconds} seconds...")
      time.sleep(retry_delay_seconds)
    except Exception as e:
      # Catch other potential errors, e.g., curl failing before JSON is returned
      print(f"Attempt {i+1}/{max_retries}: Error accessing ngrok API: {e}, retrying in {retry_delay_seconds} seconds...")
      time.sleep(retry_delay_seconds)

  if not info or 'tunnels' not in info or len(info['tunnels']) == 0:
    raise Exception("Failed to get ngrok tunnel information after multiple retries or no tunnels found.")

  in_addr = None
  out_addr = None
  for tunnel in info['tunnels']:
    url = tunnel['public_url']
    port = url.split(':')[-1]
    local_port = tunnel['config']['addr'].split(':')[-1]
    print(f'{url} -> {local_port} [{tunnel["name"]}]')
    if tunnel['name'] == 'input':
      in_addr = url
    elif tunnel['name'] == 'output':
      out_addr = url
    else:
      print(f'unknown tunnel: {tunnel["name"]}')

  if in_addr is None or out_addr is None:
    raise Exception("Could not find both input and output tunnel addresses.")

  return in_addr, out_addr

In [65]:
# Input and output ports for communication
local_in_port = 5557
local_out_port = 5558

# Start the worker


In [66]:
print('Listing contents of current directory:')
!ls -F

Listing contents of current directory:
avatarify/


In [113]:
# (Re)Start the worker with corrected paths and patched dependencies
import os
import shlex
from subprocess import Popen, PIPE
import time

# Stop any existing processes first
!pkill -f "afy/cam_fomm.py"
time.sleep(2)

# Correcting the working directory to where the actual scripts reside
work_dir = '/content/avatarify/avatarify'

# Ensure log directory exists
!mkdir -p {work_dir}/var/log

with open('/tmp/run.txt', 'w') as f:
    ps = Popen(
        shlex.split(f'./run.sh --is-worker --in-port {local_in_port} --out-port {local_out_port} --no-vcam --no-conda'),
        stdout=f, stderr=f, cwd=work_dir)
print("Worker process launched. Please check the 'Check if the worker started' cell in a few seconds.")

Worker process launched. Please check the 'Check if the worker started' cell in a few seconds.


In [99]:
# Check if the worker started (Updated path check)
!ps aux | grep 'python3 afy/cam_fomm.py' | grep -v grep | tee /tmp/ps_run
!if [[ $(cat /tmp/ps_run | wc -l) == "0" ]]; then echo "Worker failed to start"; cat /tmp/run.txt; else echo "Worker started successfully!"; fi

root       10891  112  2.5 3246400 332728 ?      Rl   09:56   0:01 python3 afy/cam_fomm.py --config fomm/config/vox-adv-256.yaml --checkpoint vox-adv-cpk.pth.tar --virt-cam 9 --relative --adapt_scale --is-worker --in-port 5557 --out-port 5558 --no-stream
Worker started successfully!


In [71]:
print('Searching for run.sh:')
!find /content -name run.sh

Searching for run.sh:
/content/avatarify/avatarify/run.sh


In [72]:
print('Listing contents of /content/avatarify:')
!ls -F /content/avatarify

Listing contents of /content/avatarify:
avatarify/


This command should print lines if the worker is successfully started

In [73]:
!ps aux | grep 'python3 afy/cam_fomm.py' | grep -v grep | tee /tmp/ps_run
!if [[ $(cat /tmp/ps_run | wc -l) == "0" ]]; then echo "Worker failed to start"; cat /tmp/run.txt; else echo "Worker started"; fi

Worker failed to start
avatarify/run.sh: line 77: scripts/settings.sh: No such file or directory
python3: can't open file '/content/avatarify/afy/cam_fomm.py': [Errno 2] No such file or directory


# Open ngrok tunnel

#### Get ngrok token
Go to https://dashboard.ngrok.com/auth/your-authtoken (sign up if required), copy your authtoken and put it below.

In [74]:
# Paste your authtoken here in quotes
authtoken = "3FfGOW9fQukubkgSyvca5uVEvwg_22NDDPAkwPykwjPQ1hLGL"

Set your region

Code | Region
--- | ---
us | United States
eu | Europe
ap | Asia/Pacific
au | Australia
sa | South America
jp | Japan
in | India

In [75]:
# Set your region here in quotes
region = "eu"

In [76]:
config =\
f"""
version: 2
authtoken: {authtoken}
region: {region}
console_ui: False
tunnels:
  input:
    addr: {local_in_port}
    proto: tcp
  output:
    addr: {local_out_port}
    proto: tcp
"""

with open('ngrok.conf', 'w') as f:
  f.write(config)


In [77]:
# This cell is no longer needed as its content has been merged into JAyPH2t2C64H.
# It can be safely ignored or removed.
# ps = Popen('./avatarify/avatarify/scripts/open_tunnel_ngrok.sh', stdout=PIPE, stderr=PIPE)
# time.sleep(3)

In [81]:
import time
import os
from subprocess import Popen, PIPE

# 1. Find the actual location of ngrok
print("Searching for ngrok executable...")
find_ngrok = !find /content -name ngrok -type f

if not find_ngrok:
    print("Ngrok not found. Re-running installation script...")
    !./scripts/get_ngrok.sh
    find_ngrok = !find /content -name ngrok -type f

if find_ngrok:
    actual_ngrok_path = find_ngrok[0]
    target_path = "/content/avatarify/ngrok"
    if actual_ngrok_path != target_path:
        print(f"Moving ngrok from {actual_ngrok_path} to {target_path}")
        !mv {actual_ngrok_path} {target_path}
        !chmod +x {target_path}
else:
    print("Critical Error: Could not find or install ngrok.")

# 2. Re-run the tunnel logic with confirmed paths
print("Opening ngrok tunnel...")
script_path = './avatarify/scripts/open_tunnel_ngrok.sh'

new_script_content = f"""#!/usr/bin/env bash
cd /content/avatarify
cmd="./ngrok start --all --config ./ngrok.conf"
kill -9 $(ps aux | grep "$cmd" | grep -v grep | awk '{{print $2}}') 2> /dev/null
echo Opening tunnel...
$cmd
"""

with open(script_path, 'w') as f:
    f.write(new_script_content)

!chmod +x {script_path}

# Run script
ps = Popen([script_path], stdout=PIPE, stderr=PIPE)
time.sleep(5)

# 3. Get addresses
try:
    in_addr, out_addr = get_tunnel_adresses()
    print("Tunnel opened successfully!")
    print(f"Input Tunnel Address: {in_addr}")
    print(f"Output Tunnel Address: {out_addr}")
except Exception as e:
    print(f"Error: {e}")
    print("Script Output:", ps.stdout.read().decode())
    print("Script Errors:", ps.stderr.read().decode())

Searching for ngrok executable...
Moving ngrok from /content/avatarify/avatarify/ngrok to /content/avatarify/ngrok
Opening ngrok tunnel...
tcp://5.tcp.eu.ngrok.io:20592 -> 5557 [input]
tcp://0.tcp.eu.ngrok.io:21748 -> 5558 [output]
Tunnel opened successfully!
Input Tunnel Address: tcp://5.tcp.eu.ngrok.io:20592
Output Tunnel Address: tcp://0.tcp.eu.ngrok.io:21748


In [82]:
print('Checking for ngrok.conf in the current directory (/content/avatarify):')
!ls -F ngrok.conf
!ls -F /content/avatarify/ngrok.conf

Checking for ngrok.conf in the current directory (/content/avatarify):
ngrok.conf
/content/avatarify/ngrok.conf


In [83]:
print('Searching for ngrok.conf in /content:')
!find /content -name ngrok.conf

Searching for ngrok.conf in /content:
/content/avatarify/ngrok.conf


In [84]:
import os

# Confirm current working directory
print(f'Current working directory: {os.getcwd()}')

# Recreate ngrok.conf (content from cell jZ5_PE_EHpCg)
# Assuming 'authtoken' and 'region' are already defined from previous cells.
# If not, please run cells YDtPpi77AkQ1 and r5e9VR9NYckJ first.

config =\
f"""
version: 2
authtoken: {authtoken}
region: {region}
console_ui: False
tunnels:
  input:
    addr: {local_in_port}
    proto: tcp
  output:
    addr: {local_out_port}
    proto: tcp
"""

with open('ngrok.conf', 'w') as f:
  f.write(config)
print('ngrok.conf has been recreated in the current working directory.')

# Verify its existence and find its absolute path
print('\nSearching for ngrok.conf again:')
!find /content -name ngrok.conf

Current working directory: /content/avatarify
ngrok.conf has been recreated in the current working directory.

Searching for ngrok.conf again:
/content/avatarify/ngrok.conf


In [85]:
!cat /content/avatarify/avatarify/avatarify/scripts/open_tunnel_ngrok.sh

cat: /content/avatarify/avatarify/avatarify/scripts/open_tunnel_ngrok.sh: No such file or directory


In [86]:
!find /content -name ngrok

/content/avatarify/ngrok


### [Optional] AWS proxy
Alternatively you can create a ssh reverse tunnel to an AWS `t3.micro` instance (it's free). It has lower latency than ngrok.

1. In your AWS console go to Services -> EC2 -> Instances -> Launch Instance;
1. Choose `Ubuntu Server 18.04 LTS` AMI;
1. Choose `t3.micro` instance type and press Review and launch;
1. Confirm your key pair and press Launch instances;
1. Go to the security group of this instance and edit inbound rules. Add TCP ports 5557 and 5558 and set Source to Anywhere. Press Save rules;
1. ssh into the instance (you can find the command in the Instances if you click on the Connect button) and add this line in the end of `/etc/ssh/sshd_config`:
```
GatewayPorts yes
```
then restart `sshd`
```
sudo service sshd restart
```
1. Copy your `key_pair.pem` by dragging and dropping it into avatarify folder in this notebook;
1. Use the command below to open the tunnel;
1. Start client with a command (substitute `run_mac.sh` with `run_windows.bat` or `run.sh`)
```
./run_mac.sh --is-client --in-addr tcp://instace.compute.amazonaws.com:5557 --out-addr tcp://instance.compute.amazonaws.com:5558
```

In [87]:
# Open reverse ssh tunnel (uncomment line below)
# !./scripts/open_tunnel_ssh.sh key_pair.pem ubuntu@instance.compute.amazonaws.com

# Start the client
When you run the cell below it will print a command. Run this command on your computer:

1. Open a terminal (in Windows open `Anaconda Prompt`);
2. Change working directory to the `avatarify` directory:</br>
* Windows (change `C:\path\to\avatarify` to your path)</br>
`cd C:\path\to\avatarify`</br></br>
* Mac/Linux (change `/path/to/avatarify` to your path)</br>
`cd /path/to/avatarify`
3. Copy-paste to the terminal the command below and run;
4. It can take some time to connect (usually up to 10 seconds). If the preview window doesn't appear in a minute or two, look for the errors above in this notebook and report in the [issues](https://github.com/alievk/avatarify/issues) or [Slack](https://join.slack.com/t/avatarify/shared_invite/zt-dyoqy8tc-~4U2ObQ6WoxuwSaWKKVOgg).

In [108]:
# Display connection commands for the user
print('Copy-paste to the terminal the command below and run (press Enter)\n')
print('Mac:')
print(f'./run_mac.sh --is-client --in-addr {in_addr} --out-addr {out_addr}')
print('\nWindows:')
print(f'run_windows.bat --is-client --in-addr {in_addr} --out-addr {out_addr}')
print('\nLinux:')
print(f'./run.sh --is-client --in-addr {in_addr} --out-addr {out_addr}')

Copy-paste to the terminal the command below and run (press Enter)

Mac:
./run_mac.sh --is-client --in-addr tcp://5.tcp.eu.ngrok.io:20592 --out-addr tcp://0.tcp.eu.ngrok.io:21748

Windows:
run_windows.bat --is-client --in-addr tcp://5.tcp.eu.ngrok.io:20592 --out-addr tcp://0.tcp.eu.ngrok.io:21748

Linux:
./run.sh --is-client --in-addr tcp://5.tcp.eu.ngrok.io:20592 --out-addr tcp://0.tcp.eu.ngrok.io:21748


# Logs

If something doesn't work as expected, please run the cells below and include the logs in your report.

In [112]:
# Comprehensive patch for face-alignment PyTorch compatibility
import face_alignment
import os
import re

utils_path = os.path.join(os.path.dirname(face_alignment.__file__), 'utils.py')

with open(utils_path, 'r') as f:
    content = f.read()

# Broadly catch the pattern t[x, y] = resolution / something
# and wrap the right side in float()
patched_content = re.sub(
    r"(t\[\d+, \d+\] = )(resolution / [hw])",
    r"\1float(\2)",
    content
)

with open(utils_path, 'w') as f:
    f.write(patched_content)

print('Applied comprehensive compatibility patch to face-alignment. Re-running worker...')


Applied comprehensive compatibility patch to face-alignment. Re-running worker...


In [110]:
print('--- Predictor Log (Last 50 lines) ---')
!cat /content/avatarify/avatarify/var/log/predictor_worker.log | tail -50

--- Predictor Log (Last 50 lines) ---
[1782468038.304392] Initialized predictor with: [[], {'config_path': 'fomm/config/vox-adv-256.yaml', 'checkpoint_path': 'vox-adv-cpk.pth.tar', 'relative': True, 'adapt_movement_scale': True, 'enc_downscale': 1}]
[1782468043.227833] predictor_worker error
[1782468043.229304] predictor_worker exit


In [111]:
print('--- Detailed System Traceback (run.txt) ---')
!cat /tmp/run.txt | tail -n 100

--- Detailed System Traceback (run.txt) ---
[1782467981.654877] Loading Predictor
[1782467983.628720] Receiving on port 5557
[1782467983.643647] Sending on port 5558
/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
[1782468038.304482] Initialized predictor with: [[], {'config_path': 'fomm/config/vox-adv-256.yaml', 'checkpoint_path': 'vox-adv-cpk.pth.tar', 'relative': True, 'adapt_movement_scale': True, 'enc_downscale': 1}]
[1782468043.227887] predictor_worker error
Traceback (most recent call last):
  File "/content/avatarify/avatarify/afy/predictor_worker.py", line 153, in predictor_worker
    result = getattr(predictor, method['name'])(*args[0], **args[1])
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

In [114]:
import time
time.sleep(5)
print('--- Final Predictor Log Check ---')
!cat /content/avatarify/avatarify/var/log/predictor_worker.log | tail -n 20
print('\n--- Final System Status ---')
!ps aux | grep 'python3 afy/cam_fomm.py' | grep -v grep

--- Final Predictor Log Check ---

--- Final System Status ---
root       22819  9.2  4.5 5463696 599144 ?      S    10:42   0:02 python3 afy/cam_fomm.py --config fomm/config/vox-adv-256.yaml --checkpoint vox-adv-cpk.pth.tar --virt-cam 9 --relative --adapt_scale --is-worker --in-port 5557 --out-port 5558 --no-stream
root       22836  0.0  2.6 5529232 346068 ?      Sl   10:42   0:00 python3 afy/cam_fomm.py --config fomm/config/vox-adv-256.yaml --checkpoint vox-adv-cpk.pth.tar --virt-cam 9 --relative --adapt_scale --is-worker --in-port 5557 --out-port 5558 --no-stream
root       22839  0.0  2.5 5463696 343784 ?      S    10:42   0:00 python3 afy/cam_fomm.py --config fomm/config/vox-adv-256.yaml --checkpoint vox-adv-cpk.pth.tar --virt-cam 9 --relative --adapt_scale --is-worker --in-port 5557 --out-port 5558 --no-stream
root       22840  0.0  2.6 5529232 346104 ?      Sl   10:42   0:00 python3 afy/cam_fomm.py --config fomm/config/vox-adv-256.yaml --checkpoint vox-adv-cpk.pth.tar --virt-cam

In [115]:
print('--- Verifying ngrok Tunnel Status ---')
!curl -s http://localhost:4040/api/tunnels | python3 -m json.tool

print('\n--- Checking for Predictor/Network Errors ---')
!tail -n 20 /content/avatarify/avatarify/var/log/predictor_worker.log
!tail -n 20 /content/avatarify/avatarify/var/log/recv_worker.log

--- Verifying ngrok Tunnel Status ---
{
    "tunnels": [
        {
            "name": "input",
            "ID": "c9b0d82764a799a17a0efd64c7dfc3a8",
            "uri": "/api/tunnels/input",
            "public_url": "tcp://5.tcp.eu.ngrok.io:20592",
            "proto": "tcp",
            "config": {
                "addr": "localhost:5557",
                "inspect": false
            },
            "metrics": {
                "conns": {
                    "count": 322,
                    "gauge": 0,
                    "rate1": 0.12390763472765828,
                    "rate5": 0.135627117795703,
                    "rate15": 0.07507532854195151,
                    "p50": 509616,
                    "p90": 615819.4,
                    "p95": 697287.4999999995,
                    "p99": 989751.0899999968
                },
                "http": {
                    "count": 0,
                    "rate1": 0,
                    "rate5": 0,
                    "rate15": 0,
    

In [116]:
print('--- Resetting ngrok Tunnels ---')
!pkill ngrok
import time
time.sleep(2)
# Restarting tunnel logic
ps = Popen([script_path], stdout=PIPE, stderr=PIPE)
time.sleep(5)
try:
    in_addr, out_addr = get_tunnel_adresses()
    print(f'NEW Input: {in_addr}')
    print(f'NEW Output: {out_addr}')
    print('\nUse these new addresses in your Windows command.')
except Exception as e:
    print(f'Error: {e}')

--- Resetting ngrok Tunnels ---
tcp://5.tcp.eu.ngrok.io:20592 -> 5557 [input]
tcp://4.tcp.eu.ngrok.io:28558 -> 5558 [output]
NEW Input: tcp://5.tcp.eu.ngrok.io:20592
NEW Output: tcp://4.tcp.eu.ngrok.io:28558

Use these new addresses in your Windows command.


In [117]:
print('--- Checking Predictor Health for Face Detection ---')
!tail -n 50 /content/avatarify/avatarify/var/log/predictor_worker.log
print('\n--- Checking for Input Stream Activity ---')
!tail -n 20 /content/avatarify/avatarify/var/log/recv_worker.log

--- Checking Predictor Health for Face Detection ---
[1782470769.014286] Initialized predictor with: [[], {'config_path': 'fomm/config/vox-adv-256.yaml', 'checkpoint_path': 'vox-adv-cpk.pth.tar', 'relative': True, 'adapt_movement_scale': True, 'enc_downscale': 1}]

--- Checking for Input Stream Activity ---
[1782470543.062140] Receiving on port 5557


In [118]:
print('--- Force Refreshing Worker and Ports ---')
!pkill -f "python3 afy/cam_fomm.py"
import time
time.sleep(2)

# Restart with explicit log clearing
work_dir = '/content/avatarify/avatarify'
!rm -f {work_dir}/var/log/*.log

from subprocess import Popen
with open('/tmp/run.txt', 'w') as f:
    ps = Popen(
        ['./run.sh', '--is-worker', '--in-port', '5557', '--out-port', '5558', '--no-vcam', '--no-conda'],
        stdout=f, stderr=f, cwd=work_dir)

time.sleep(5)
print('Worker restarted. Please check if your Windows client starts showing frames.')
!ps aux | grep 'python3 afy/cam_fomm.py' | grep -v grep

--- Force Refreshing Worker and Ports ---
Worker restarted. Please check if your Windows client starts showing frames.
root       30782 52.2  4.5 5463428 598432 ?      S    11:13   0:02 python3 afy/cam_fomm.py --config fomm/config/vox-adv-256.yaml --checkpoint vox-adv-cpk.pth.tar --virt-cam 9 --relative --adapt_scale --is-worker --in-port 5557 --out-port 5558 --no-stream
root       30803  0.0  2.6 5528964 346048 ?      Sl   11:13   0:00 python3 afy/cam_fomm.py --config fomm/config/vox-adv-256.yaml --checkpoint vox-adv-cpk.pth.tar --virt-cam 9 --relative --adapt_scale --is-worker --in-port 5557 --out-port 5558 --no-stream
root       30806  0.0  2.5 5463428 343516 ?      S    11:13   0:00 python3 afy/cam_fomm.py --config fomm/config/vox-adv-256.yaml --checkpoint vox-adv-cpk.pth.tar --virt-cam 9 --relative --adapt_scale --is-worker --in-port 5557 --out-port 5558 --no-stream
root       30807  0.0  2.6 5528964 346152 ?      Sl   11:13   0:00 python3 afy/cam_fomm.py --config fomm/config/vox-

In [119]:
print('--- Restarting Worker Process ---')
!pkill -f "python3 afy/cam_fomm.py"
import time
time.sleep(2)

# Clear logs to have a fresh view of detection status
work_dir = '/content/avatarify/avatarify'
!rm -f {work_dir}/var/log/*.log

from subprocess import Popen
with open('/tmp/run.txt', 'w') as f:
    ps = Popen(
        ['./run.sh', '--is-worker', '--in-port', '5557', '--out-port', '5558', '--no-vcam', '--no-conda'],
        stdout=f, stderr=f, cwd=work_dir)

time.sleep(5)
print('Worker restarted successfully. You can now try to connect your local client again.')
!ps aux | grep 'python3 afy/cam_fomm.py' | grep -v grep

--- Restarting Worker Process ---
Worker restarted successfully. You can now try to connect your local client again.
root       31332 53.0  4.5 5463556 598948 ?      S    11:15   0:02 python3 afy/cam_fomm.py --config fomm/config/vox-adv-256.yaml --checkpoint vox-adv-cpk.pth.tar --virt-cam 9 --relative --adapt_scale --is-worker --in-port 5557 --out-port 5558 --no-stream
root       31346  0.0  2.6 5529092 346016 ?      Sl   11:15   0:00 python3 afy/cam_fomm.py --config fomm/config/vox-adv-256.yaml --checkpoint vox-adv-cpk.pth.tar --virt-cam 9 --relative --adapt_scale --is-worker --in-port 5557 --out-port 5558 --no-stream
root       31349  0.0  2.5 5463556 343612 ?      S    11:15   0:00 python3 afy/cam_fomm.py --config fomm/config/vox-adv-256.yaml --checkpoint vox-adv-cpk.pth.tar --virt-cam 9 --relative --adapt_scale --is-worker --in-port 5557 --out-port 5558 --no-stream
root       31350  0.0  2.6 5529092 346124 ?      Sl   11:15   0:00 python3 afy/cam_fomm.py --config fomm/config/vox-ad

In [121]:
print('--- Current Predictor Worker Log ---')
!cat /content/avatarify/avatarify/var/log/predictor_worker.log | tail -n 50

--- Current Predictor Worker Log ---
[1782472577.965846] Initialized predictor with: [[], {'config_path': 'fomm/config/vox-adv-256.yaml', 'checkpoint_path': 'vox-adv-cpk.pth.tar', 'relative': True, 'adapt_movement_scale': True, 'enc_downscale': 1}]


### Performance & Lag Troubleshooting
Run the following cells to check if the lag is network-related or hardware-related.

In [122]:
import time
import torch

print(f"--- GPU Status ---")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: GPU NOT DETECTED. The server is running on CPU which will cause extreme lag.")

print(f"\n--- Network Latency (Internal) ---")
!ping -c 4 google.com

--- GPU Status ---
CUDA Available: True
Device Name: Tesla T4

--- Network Latency (Internal) ---
/bin/bash: line 1: ping: command not found


In [123]:
print('--- Checking Frame Processing Speed ---')
# This looks for lines indicating how long the predictor takes per frame
!grep "predictor" /content/avatarify/avatarify/var/log/predictor_worker.log | tail -n 20

print('\n--- If you are NOT in Europe, change the region in cell r5e9VR9NYckJ and restart ---')

--- Checking Frame Processing Speed ---
[1782472577.965846] Initialized predictor with: [[], {'config_path': 'fomm/config/vox-adv-256.yaml', 'checkpoint_path': 'vox-adv-cpk.pth.tar', 'relative': True, 'adapt_movement_scale': True, 'enc_downscale': 1}]

--- If you are NOT in Europe, change the region in cell r5e9VR9NYckJ and restart ---


In [124]:
print('--- Monitoring Input Stream (Run this while your client is connected) ---')
# This checks if the receiver is getting packets
!tail -n 20 /content/avatarify/avatarify/var/log/recv_worker.log

print('\n--- Checking for processing errors ---')
!tail -n 20 /content/avatarify/avatarify/var/log/predictor_worker.log

--- Monitoring Input Stream (Run this while your client is connected) ---
[1782472516.748371] Receiving on port 5557

--- Checking for processing errors ---
[1782472577.965846] Initialized predictor with: [[], {'config_path': 'fomm/config/vox-adv-256.yaml', 'checkpoint_path': 'vox-adv-cpk.pth.tar', 'relative': True, 'adapt_movement_scale': True, 'enc_downscale': 1}]


In [125]:
print('--- Real-time Activity Monitor ---')
print('If you see no new lines below while the client is running, the connection is blocked.')
!tail -f /content/avatarify/avatarify/var/log/predictor_worker.log | grep -i "frame"

--- Real-time Activity Monitor ---
If you see no new lines below while the client is running, the connection is blocked.
^C


In [126]:
print('--- Detection Status Check ---')
# This checks the last 50 lines for face detection failures
!grep -i "None" /content/avatarify/avatarify/var/log/predictor_worker.log | tail -n 20

print('\n--- Network Health (Total Connections) ---')
!curl -s http://localhost:4040/api/tunnels | grep -o '"count":[0-9]*' | head -n 2

--- Detection Status Check ---

--- Network Health (Total Connections) ---


In [127]:
print('--- Real-time Latency & Detection Monitor ---')
print('Monitoring the last 20 frames...')
!tail -n 20 /content/avatarify/avatarify/var/log/predictor_worker.log | grep -E "frame|None|predict"

--- Real-time Latency & Detection Monitor ---
Monitoring the last 20 frames...
[1782472577.965846] Initialized predictor with: [[], {'config_path': 'fomm/config/vox-adv-256.yaml', 'checkpoint_path': 'vox-adv-cpk.pth.tar', 'relative': True, 'adapt_movement_scale': True, 'enc_downscale': 1}]
[1782473296.816449] predictor_worker: user interrupt
[1782473296.816521] predictor_worker exit


In [ ]:
print('--- Restarting Worker & Monitoring Detection ---')
!pkill -f "python3 afy/cam_fomm.py"
import time
time.sleep(2)

# Restart worker
work_dir = '/content/avatarify/avatarify'
from subprocess import Popen
with open('/tmp/run.txt', 'w') as f:
    ps = Popen(['./run.sh', '--is-worker', '--in-port', '5557', '--out-port', '5558', '--no-vcam', '--no-conda'],
               stdout=f, stderr=f, cwd=work_dir)

print('Worker restarted. Please connect your client and watch the output below.')
print('If you see many "None" lines, the AI cannot see your face.')
!tail -f /content/avatarify/avatarify/var/log/predictor_worker.log | grep -E "None|frame"

--- Restarting Worker & Monitoring Detection ---
Worker restarted. Please connect your client and watch the output below.
If you see many "None" lines, the AI cannot see your face.
tail: /content/avatarify/avatarify/var/log/predictor_worker.log: file truncated


### If detection still fails:
Try to align your face exactly with the 'X' or guide in the Avatarify window on your computer. If the server logs above show many `None` returns, try moving the camera closer.

**How to reduce lag:**
1. **Use a closer region**: You are currently using `eu`. If you are in the US or Asia, change the `region` variable in the ngrok setup cell to `us` or `ap` and restart the tunnels.
2. **Check Resolution**: Ensure your webcam resolution on the local client is set to 256x256 or similar. High-resolution input slows down processing.
3. **Switch to SSH**: As mentioned in the notebook, ngrok can add 100ms-1000ms of lag. If the GPU is working, the lag is likely the tunnel.

In [100]:
#@title View Receiver Log
!cat /content/avatarify/avatarify/var/log/recv_worker.log | tail -100

[1782467795.709308] Receiving on port 5557
[1782467803.350499] recv_worker exit


In [101]:
#@title View Predictor Log
!cat /content/avatarify/avatarify/var/log/predictor_worker.log | tail -100

[1782467798.384939] Initialized predictor with: [[], {'config_path': 'fomm/config/vox-adv-256.yaml', 'checkpoint_path': 'vox-adv-cpk.pth.tar', 'relative': True, 'adapt_movement_scale': True, 'enc_downscale': 1}]
[1782467802.959073] predictor_worker error
[1782467802.960577] predictor_worker exit


In [96]:
print('Searching for Avatarify log files...')
!find /content -name "*.log"

Searching for Avatarify log files...
/content/.config/logs/2026.06.04/13.39.22.644527.log
/content/.config/logs/2026.06.04/13.39.24.274402.log
/content/.config/logs/2026.06.04/13.39.12.467650.log
/content/.config/logs/2026.06.04/13.38.54.010366.log
/content/.config/logs/2026.06.04/13.39.37.138439.log
/content/.config/logs/2026.06.04/13.39.36.079406.log


In [102]:
#@title View Sender Log
!cat /content/avatarify/avatarify/var/log/send_worker.log | tail -100

[1782467795.723135] Sending on port 5558
[1782467803.390391] send_worker exit


In [103]:
import os
checkpoint_file = 'vox-adv-cpk.pth.tar'
work_dir = '/content/avatarify/avatarify'

print(f'Checking for {checkpoint_file} in {work_dir}...')
if os.path.exists(os.path.join(work_dir, checkpoint_file)):
    print('Checkpoint found in work_dir.')
else:
    print('Checkpoint NOT found in work_dir. Searching /content...')
    found_paths = !find /content -name {checkpoint_file}
    if found_paths:
        print(f'Found at: {found_paths[0]}')
        print(f'Moving to {work_dir}...')
        !mv {found_paths[0]} {work_dir}/
    else:
        print('Checkpoint not found anywhere. Please re-run the download cell.')

Checking for vox-adv-cpk.pth.tar in /content/avatarify/avatarify...
Checkpoint found in work_dir.
